# Diamond frontier4 smoke: frontier, 10 minutes

WikiText-only smoke test for the Kaggle GPU path. This checks model memory and
throughput before spending a full session. It uses a 10-minute training budget.


In [ ]:
#@title GPU / TensorFlow check
!nvidia-smi || true
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title Clone branch
%cd /kaggle/working
!rm -rf nanlm_experimental
!git clone --branch master --depth 1 https://github.com/shemshallah/nanlm_experimental.git
%cd /kaggle/working/nanlm_experimental
!git log --oneline -1


In [ ]:
#@title WikiText-103 download + article split
!if [ ! -s /kaggle/working/wiki.train.raw ]; then \
  wget -q --show-progress -O /kaggle/working/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!python diamond/colab/prepare_wikitext.py \
  --raw /kaggle/working/wiki.train.raw \
  --out /kaggle/working/corpus \
  --max-articles 4000 \
  --max-chars 20000000


In [ ]:
#@title 10-minute smoke train (frontier)
!STATE_DIR=/kaggle/working/smoke_frontier \
 CORPUS_DIR=/kaggle/working/corpus \
 MODEL_SIZE=frontier BATCH=32 EVAL_BATCH=32 \
 EPOCHS=1 EXAMPLES=1000 CORPUS_CHARS=5000000 \
 TIME_BUDGET=10 TEST_SIZE=256 \
 bash diamond/kaggle/train_kaggle.sh


In [ ]:
#@title Report
!python diamond/colab/report_metrics.py /kaggle/working/smoke_frontier/diamond_metrics.json
